# Event Labeling

## Overview

This notebook demonstrates the public functions in `event_labeling` using dollar bars built from a compact synthetic trade stream.
- Problem: fixed-horizon labels ignore whether a trade first reaches its profit target or stop loss.
- Approach: select candidate events, set volatility-scaled barriers, and label the first barrier touched.
- Daily Volatility: It estimates the target return used to scale horizontal barriers.
- Triple-Barrier Events: It constructs event horizons with profit-taking, stop-loss, and vertical barriers.
- Label Event Outcomes: It converts barrier hits into discrete labels and removes rare classes.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
from src.data_preprocessing.event_labeling import (
    apply_profit_taking_stop_loss_on_t1,
    drop_labels,
    get_bins,
    get_daily_volatility,
    get_events,
    get_vertical_barriers,
)
from src.data_preprocessing.market_structured_bars import get_dollar_bars

## Define Synthetic Trades

This cell defines the reproducible intraday trade stream and resamples it into dollar bars.
- dollar_close is the timestamped price series used by the labeling functions.
- target_bars_per_day sets the dollar-bar threshold from the synthetic trading-day count.


In [ ]:
rng = np.random.default_rng(42)
trading_days = pd.bdate_range("2024-01-02", periods=20, tz="UTC")
trades_per_day = 48
minutes_from_open = np.tile(np.arange(trades_per_day) * 5 + 14 * 60 + 30, len(trading_days))
timestamps = trading_days.repeat(trades_per_day) + pd.to_timedelta(minutes_from_open, unit="m")
prices = 180.0 * np.exp(np.cumsum(rng.normal(0.00005, 0.0015, len(timestamps))))
sizes = rng.lognormal(3.7, 0.55, len(timestamps)).round().astype(int)
trades = pd.DataFrame({"timestamp": timestamps, "symbol": "AAPL", "price": prices, "size": sizes})
notional = trades["price"].astype(float) * trades["size"].astype(float)
target_bars_per_day = 24
trading_days = len(trading_days)
target_num_bars = max(1, trading_days * target_bars_per_day)
dollar_threshold = float(notional.sum() / target_num_bars)
dollar_bars = get_dollar_bars(trades, threshold=dollar_threshold).ohlcv
dollar_close = dollar_bars["close"].astype(float)

print("source: synthetic intraday trade stream")
print(f"trading_days: {trading_days}")
print(f"target_bars_per_day: {target_bars_per_day}")
print(f"num_dollar_bars: {len(dollar_close):,}")
dollar_close.head()

## Estimate Daily Volatility

This cell estimates the volatility target from the dollar-bar close series.
- span0 controls the exponentially weighted lookback used by get_daily_volatility.
- The resulting series scales each event's profit-taking and stop-loss barriers.


In [ ]:
daily_volatility = get_daily_volatility(dollar_close, span0=50)
daily_volatility.dropna().head()

## Build Triple-Barrier Events

This cell constructs triple-barrier events from candidate timestamps, volatility targets, and a side signal.
- `t_events` contains every fifth dollar-bar timestamp as a reproducible placeholder for external events such as news releases.
- t1 is the vertical barrier ten dollar bars after the event start.
- pt_sl=[1.0, 1.0] sets symmetric volatility-scaled profit-taking and stop-loss barriers.
- The plot follows AFML Figure 3.1 by showing the three barrier types for selected events.


In [ ]:
event_spacing = 5
t_events = pd.DatetimeIndex(dollar_close.index[::event_spacing])
t1 = get_vertical_barriers(t_events, dollar_close, num_bars=10)
targets = daily_volatility.reindex(t_events)
side = dollar_close.pct_change(5).reindex(t_events).apply(lambda value: 1.0 if value >= 0 else -1.0)

eligible_events = targets.dropna().index.intersection(side.dropna().index).intersection(t1.index)
min_return = float(targets.loc[eligible_events].quantile(0.25))

events = get_events(
    close=dollar_close,
    t_events=eligible_events,
    pt_sl=[1.0, 1.0],
    trgt=targets,
    min_ret=min_return,
    num_threads=1,
    t1=t1,
    side=side,
)

events.head()


This cell plots selected event paths with their profit-taking, stop-loss, and vertical barriers.


In [ ]:
plot_events = events.head(3)
window_start = plot_events.index.min()
window_end = plot_events["t1"].max()
plot_close = dollar_close.loc[window_start:window_end]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_close.index, plot_close, color="tab:blue", linewidth=1.1, label="Dollar-bar close")
ax.scatter(plot_events.index, dollar_close.reindex(plot_events.index), color="tab:red", s=24, label="Event start")

for index, (event_time, event_row) in enumerate(plot_events.iterrows()):
    event_price = dollar_close.loc[event_time]
    upper_barrier = event_price * (1 + event_row["trgt"])
    lower_barrier = event_price * (1 - event_row["trgt"])
    barrier_time = event_row["t1"]
    ax.hlines(upper_barrier, event_time, barrier_time, color="tab:green", linewidth=1.2, label="Profit-taking barrier" if index == 0 else None)
    ax.hlines(lower_barrier, event_time, barrier_time, color="tab:purple", linewidth=1.2, label="Stop-loss barrier" if index == 0 else None)
    ax.vlines(barrier_time, lower_barrier, upper_barrier, color="tab:orange", linewidth=1.0, label="Vertical barrier" if index == 0 else None)

ax.set_title("AFML Figure 3.1: Triple-barrier configurations")
ax.set_xlabel("time")
ax.set_ylabel("close")
ax.grid(alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()


## Label Event Outcomes

This cell converts barrier hits into realized returns and discrete labels.
- apply_profit_taking_stop_loss_on_t1 finds the first horizontal barrier touched before t1.
- get_bins produces side-adjusted returns and labels.
- drop_labels removes label classes below the minimum frequency threshold.


This cell reports the first profit-taking and stop-loss barrier hit for each event.


In [ ]:
barrier_hits = apply_profit_taking_stop_loss_on_t1(dollar_close, events, pt_sl=[1.0, 1.0], molecule=events.index)
barrier_hits.head()

This cell converts the completed event paths into side-adjusted returns and labels.


In [ ]:
labeled_events = events.copy()
labeled_events["t1"] = barrier_hits.apply(
    lambda row: min([value for value in row if pd.notna(value)]) if any(pd.notna(row)) else pd.NaT,
    axis=1,
).fillna(events["t1"])
bins = get_bins(labeled_events, dollar_close)
bins.head()

This cell compares the label frequencies before and after rare-label filtering.


In [ ]:
filtered_bins = drop_labels(bins.copy(), min_pct=0.10)

label_summary = pd.concat(
    {
        "raw_label_frequency": bins["bin"].value_counts(normalize=True).sort_index(),
        "filtered_label_frequency": filtered_bins["bin"].value_counts(normalize=True).sort_index(),
    },
    axis=1,
)

label_summary